In [1]:
import pandas as pd 
multi_df = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/FewRel/Train/MINT/FewRel_full_binary_question_result.csv')

In [2]:
multi_df.columns[3:]

Index(['text', 'subject', 'object', 'subj_start', 'subj_end', 'obj_start',
       'obj_end', 'subj_id', 'obj_id', 'token',
       ...
       'associated_with_region', 'artist_origin_city',
       'reorganized_after_loss_of_papal_states',
       'held_diplomatic_position_in_location', 'partnership_location',
       'diplomatic_location_request', 'affiliated_with_location',
       'participant_at_event_location', 'artist_origin_location',
       'inventor_location_use'],
      dtype='object', length=1826)

In [3]:
multi_df.head(2)

,id,relation,relation_id,text,subject,object,subj_start,subj_end,obj_start,obj_end,...,associated_with_region,artist_origin_city,reorganized_after_loss_of_papal_states,held_diplomatic_position_in_location,partnership_location,diplomatic_location_request,affiliated_with_location,participant_at_event_location,artist_origin_location,inventor_location_use
0,train_P1877_557,after a work by,P1877,"After reading Alan Glynn 's "" The Dark Fields ...",limitless,alan glynn,14,14,2,3,...,0,0,0,0,0,0,0,0,0,0
1,train_P1877_158,after a work by,P1877,"It was a sequel to the 1952 film "" Heidi "" , w...",heidi,johanna spyri,9,9,25,26,...,0,0,0,0,0,0,0,0,0,0


In [4]:
import ast 

multi_df = multi_df.rename(columns={'gpt4_phrase': 'predicted_relation'})

multi_df['multi_labels'] = multi_df['multi_labels'].apply(ast.literal_eval) 
labels_series = multi_df['multi_labels'].explode() 
labels_count = labels_series.value_counts().reset_index() 
labels_count.columns = ['predicted_relation', 'num_samples'] 

relation_to_ids = ( 
    multi_df
        .explode('multi_labels')                        
        .groupby('multi_labels')['id']                  
        .apply(list)                                    
        .to_dict()                                      
)


labels_count['id_list'] = labels_count['predicted_relation'].map(relation_to_ids) 

relations = labels_count['predicted_relation'].tolist() 


for rel in relations: 
    labels_count[rel] = 0
    

id_sets = [set(ids) for ids in labels_count['id_list']]


for i, rel_i in enumerate(relations): 
    ids_i = id_sets[i] 
    for j, rel_j in enumerate(relations): 
        ids_j = id_sets[j] 
        overlap = len(ids_i & ids_j)   
        labels_count.at[i, rel_j] = overlap
labels_count.head(2)

/tmp/ipykernel_1265871/2021611887.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  labels_count[rel] = 0
/tmp/ipykernel_1265871/2021611887.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  labels_count[rel] = 0
/tmp/ipykernel_1265871/2021611887.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()

,predicted_relation,num_samples,id_list,ownership_or_affiliation,attribute-of,affiliation_association,participating_entity,is_affiliated_with,property_of,affiliated_with,...,spouse_of_mayor_of,located_in_desert,assassination_impact_on_cultural_identity,threatens_protection_of_religious_rights,has_flagship_store_in,capital_city_of_country,federal_republic_of_germany_contains_state,formed_by_glacial_activity_of,named_after_fortress_in_defense_line,reorganized_after_loss_of_papal_states
0,ownership_or_affiliation,2867,"[train_P1877_63, train_P1877_627, train_P1877_...",2867,2209,2261,1966,2084,1948,2047,...,0,1,1,1,1,1,1,0,1,1
1,attribute-of,2592,"[train_P1877_158, train_P1877_63, train_P1877_...",2209,2592,1911,1801,1764,2118,1791,...,0,1,1,1,1,1,1,0,1,0


In [5]:
labels_count.to_csv('/workspace/KG-known0/ARR2026/Data_paper/FewRel/Train/MINT/FewRel_full_intersection_matrix.csv', index=False)